# 7043SCN Task 2 — Chef’s Hat Gym (Variants 2/3 & 6)

**Student ID:** _REPLACE_ME_  
**Variant(s):** Variant 2/3 (Sparse/Delayed Reward via REINFORCE) + Variant 6 (Generative AI Augmentation via Behaviour Cloning)

## How to run (Colab)
1. Run cells top-to-bottom.
2. If execution is slow, reduce `N_TRAIN_GAMES`, `N_DATA_GAMES`, and `N_EVAL_GAMES`.
3. Outputs are saved as PNG + JSON under `task2_outputs/` and `task2_png_outputs/`.


In [ ]:
!pip -q install chefshatgym==3.0.0.1 gym==0.26.2
!pip -q install numpy matplotlib torch pillow
print("✅ Install complete")

## 1. Imports, seeds, and helpers

In [ ]:
import os, sys, time, json, logging
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

import torch
import torch.nn as nn
import torch.optim as optim

# ChefsHatGym (per official docs imports)
from ChefsHatGym.gameRooms.chefs_hat_room_local import ChefsHatRoomLocal
from ChefsHatGym.env import ChefsHatEnv
from ChefsHatGym.agents.agent_random import AgentRandon

# quiet logs
logging.getLogger().setLevel(logging.ERROR)
for name in list(logging.root.manager.loggerDict):
    if name.startswith("ROOM_") or name.startswith("PLAYER_"):
        logging.getLogger(name).setLevel(logging.ERROR)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def extract_perf_scores(info):
    """
    Handles different info dict keys across versions.
    Returns list[float] performance per player in seating order.
    """
    for k in ["Game_Performance_Score", "performanceScore", "Performance_Score", "PerformanceScore"]:
        if k in info:
            v = info[k]
            # sometimes dict values may be list/np array
            return [float(x) for x in list(v)]
    # fallback: if only score/points exist
    for k in ["score", "Score", "points", "Points"]:
        if k in info:
            v = info[k]
            return [float(x) for x in list(v)]
    raise KeyError(f"Could not find performance key in info. Keys found: {list(info.keys())[:30]}...")

def softmax_np(x):
    x = x - np.max(x)
    e = np.exp(x)
    return e / (e.sum() + 1e-12)

def last200_mask(observation, n_actions=200):
    """
    Docs: observation = [hand + field + possibleActions]; possibleActions is length 200.
    We safely try to read the last 200 values as a 0/1 mask.
    """
    obs = np.asarray(observation, dtype=np.float32).ravel()
    if obs.shape[0] < n_actions:
        return None
    mask = obs[-n_actions:]
    # accept “almost binary”
    if mask.sum() <= 0:
        return None
    if mask.min() < -1e-3:
        return None
    return (mask > 0.5).astype(np.float32)

## 2. Environment runners + Baseline (Random)

In [ ]:
def run_game(players, stop_criteria=1, room_name="room_eval"):
    room = ChefsHatRoomLocal(
        room_name=room_name,
        game_type=ChefsHatEnv.GAMETYPE["MATCHES"],
        stop_criteria=stop_criteria,
    )
    for p in players:
        room.add_player(p)
    return room.start_new_game()

def eval_agent(agent_factory, n_games=10):
    scores = []
    for g in range(n_games):
        p0 = agent_factory("AGENT")
        others = [AgentRandon(name=f"R{i}") for i in range(1, 4)]
        info = run_game([p0] + others, stop_criteria=1, room_name=f"eval_{g}")
        perf = extract_perf_scores(info)
        scores.append(perf[0])  # our agent is index 0
    return scores, float(np.mean(scores)), float(np.std(scores))

N_EVAL_GAMES = 10

t0 = time.time()
baseline_scores, baseline_mean, baseline_std = eval_agent(lambda name: AgentRandon(name=name), n_games=N_EVAL_GAMES)
print("Baseline mean:", baseline_mean)
print("Baseline std :", baseline_std)
print("Seconds:", round(time.time()-t0, 2))

## 3. Policy network + masked action sampling

In [ ]:
N_ACTIONS = 200

class PolicyNet(nn.Module):
    def __init__(self, input_dim, hidden=256, n_actions=200):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_actions)
        )
    def forward(self, x):
        return self.net(x)

def masked_categorical_sample(logits_np, mask_np, temperature=1.0):
    """
    logits_np: (200,)
    mask_np : (200,) 0/1
    """
    if mask_np is None:
        # no mask => uniform over all actions
        probs = softmax_np(logits_np / max(temperature, 1e-6))
        a = int(np.random.choice(len(probs), p=probs))
        return a, probs[a]

    valid = np.where(mask_np > 0.5)[0]
    if len(valid) == 0:
        a = int(np.random.randint(0, len(logits_np)))
        return a, 1.0/len(logits_np)

    v_logits = (logits_np[valid] / max(temperature, 1e-6))
    v_probs = softmax_np(v_logits)
    a = int(np.random.choice(valid, p=v_probs))
    # return chosen prob in full space terms
    return a, float(v_probs[np.where(valid == a)[0][0]])

## 4. Variant 2/3 — REINFORCE agent (delayed reward)

In [ ]:
class DelayedRewardREINFORCEAgent(AgentRandon):
    """
    Variant 2/3: reward is sparse/delayed (only after match/game).
    We do REINFORCE: loss = -R * sum(log pi(a_t|s_t)).
    """
    def __init__(self, name, policy, temperature=1.0):
        super().__init__(name=name)
        self.policy = policy
        self.temperature = temperature
        self.saved_logprobs = []
        self.saved_obs = []

    def reset_episode(self):
        self.saved_logprobs = []
        self.saved_obs = []

    def get_action(self, observation):
        obs = np.asarray(observation, dtype=np.float32).ravel()
        obs_t = torch.tensor(obs, device=device).unsqueeze(0)

        with torch.no_grad():
            logits = self.policy(obs_t).squeeze(0).detach().cpu().numpy()

        mask = last200_mask(observation, n_actions=N_ACTIONS)
        a, _ = masked_categorical_sample(logits, mask, temperature=self.temperature)

        # store logprob computed in torch for stability
        logits_t = self.policy(obs_t).squeeze(0)
        if mask is not None:
            # mask invalid actions by -inf
            mask_t = torch.tensor(mask, device=device)
            masked_logits = logits_t + (mask_t - 1.0) * 1e9  # invalid -> very negative
            dist = torch.distributions.Categorical(logits=masked_logits)
        else:
            dist = torch.distributions.Categorical(logits=logits_t)

        a_t = torch.tensor(a, device=device)
        self.saved_logprobs.append(dist.log_prob(a_t))
        self.saved_obs.append(obs)

        onehot = [0]*N_ACTIONS
        onehot[a] = 1
        return onehot

## 5. Train REINFORCE + evaluate

In [ ]:
N_TRAIN_GAMES = 10  # increase for better learning (slower)
LR = 3e-4
TEMPERATURE = 0.9
GAMMA = 1.0  # fully delayed: no step reward; keep 1.0
PRINT_EVERY = 10

# Need input_dim: run 1 short game to get an observation
tmp_info = run_game([AgentRandon("A0"), AgentRandon("A1"), AgentRandon("A2"), AgentRandon("A3")], stop_criteria=1, room_name="tmp_dim")
# We can’t read obs from info; instead we run a room with a logger agent once:
class ObsProbe(AgentRandon):
    def __init__(self, name):
        super().__init__(name=name)
        self.first_obs = None
    def get_action(self, observation):
        if self.first_obs is None:
            self.first_obs = np.asarray(observation, dtype=np.float32).ravel()
        return super().get_action(observation)

probe = ObsProbe("PROBE")
info = run_game([probe, AgentRandon("R1"), AgentRandon("R2"), AgentRandon("R3")], stop_criteria=1, room_name="probe_room")
input_dim = int(probe.first_obs.shape[0])
print("Detected input_dim:", input_dim)

policy = PolicyNet(input_dim=input_dim, hidden=256, n_actions=N_ACTIONS).to(device)
opt = optim.Adam(policy.parameters(), lr=LR)

train_returns = []
eval_means = []

def train_one_game(game_idx):
    agent = DelayedRewardREINFORCEAgent("LEARNER", policy, temperature=TEMPERATURE)
    agent.reset_episode()
    others = [AgentRandon(name=f"R{i}") for i in range(1, 4)]
    info = run_game([agent] + others, stop_criteria=1, room_name=f"train_{game_idx}")
    perf = extract_perf_scores(info)[0]  # delayed reward signal

    # REINFORCE update
    R = torch.tensor(perf, device=device, dtype=torch.float32)
    loss = -R * torch.stack(agent.saved_logprobs).sum()

    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
    opt.step()
    return float(perf), float(loss.detach().cpu().item())

t0 = time.time()
for g in range(1, N_TRAIN_GAMES+1):
    R, L = train_one_game(g)
    train_returns.append(R)
    if g % PRINT_EVERY == 0:
        scores, m, s = eval_agent(lambda name: DelayedRewardREINFORCEAgent(name, policy, temperature=0.7), n_games=8)
        eval_means.append((g, m, s))
        print(f"[{g}/{N_TRAIN_GAMES}] train_R={np.mean(train_returns[-10:]):.3f}  eval_mean={m:.3f}±{s:.3f}")

print("Training seconds:", round(time.time()-t0, 2))

## 6. Plots (training + eval)

In [ ]:
# Safe plotting (won't crash if training was interrupted)
import matplotlib.pyplot as plt
import numpy as np

if 'train_returns' in globals() and len(train_returns) > 0:
    plt.figure(figsize=(6,3.2))
    plt.plot(train_returns)
    plt.xlabel("Training game")
    plt.ylabel("Delayed reward (performance)")
    plt.title("Variant 2/3: REINFORCE training rewards")
    plt.grid(True)
    plt.tight_layout()
    plt.show()
else:
    print("⚠ train_returns not found or empty. Run the training cell above.")

if 'eval_means' in globals() and len(eval_means) > 0:
    xs = [a for (a,_,_) in eval_means]
    ms = [b for (_,b,_) in eval_means]
    ss = [c for (_,_,c) in eval_means]
    plt.figure(figsize=(6,3.2))
    plt.errorbar(xs, ms, yerr=ss, marker="o", capsize=4)
    plt.xlabel("Training game")
    plt.ylabel("Eval mean performance")
    plt.title("Evaluation during training")
    plt.grid(True)
    plt.tight_layout()
    plt.show()
else:
    print("ℹ eval_means not found/empty (this is OK if you disabled periodic eval).")


## 7. Variant 6 — Collect “best random” trajectories

In [ ]:
MAX_SAMPLES_PER_GAME = 50

class LoggingRandom(AgentRandon):
    def __init__(self, name):
        super().__init__(name=name)
        self.obs_log = []
        self.act_log = []

    def reset_logs(self):
        self.obs_log, self.act_log = [], []

    def get_action(self, observation):
        action_onehot = super().get_action(observation)
        a = int(np.argmax(action_onehot))
        self.obs_log.append(np.asarray(observation, dtype=np.float32).ravel())
        self.act_log.append(a)
        return action_onehot

def collect_best_player_data(n_games=12):
    X, y = [], []
    for g in range(n_games):
        agents = [LoggingRandom(name=f"L{i}") for i in range(4)]
        for a in agents:
            a.reset_logs()

        info = run_game(agents, stop_criteria=1, room_name=f"data_{g}")
        perf = extract_perf_scores(info)
        best_idx = int(np.argmax(perf))

        steps = min(len(agents[best_idx].obs_log), MAX_SAMPLES_PER_GAME)
        if steps > 0:
            X.extend(agents[best_idx].obs_log[:steps])
            y.extend(agents[best_idx].act_log[:steps])

        print(f"Game {g+1}: best={best_idx} perf={perf[best_idx]:.3f} samples+={steps} total={len(X)}")
        sys.stdout.flush()

    if len(X) == 0:
        raise RuntimeError("No samples collected (unexpected).")

    X = np.stack(X).astype(np.float32)
    y = np.array(y, dtype=np.int64)
    return X, y

N_DATA_GAMES = 10
X, y = collect_best_player_data(n_games=N_DATA_GAMES)
print("X:", X.shape, "y:", y.shape)

## 8. Train Behaviour Cloning prior

In [ ]:
PRIOR_EPOCHS = 4
PRIOR_BATCH = 256
PRIOR_LR = 8e-4

prior = PolicyNet(input_dim=X.shape[1], hidden=256, n_actions=N_ACTIONS).to(device)
opt_prior = optim.Adam(prior.parameters(), lr=PRIOR_LR, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()

X_t = torch.tensor(X, device=device)
y_t = torch.tensor(y, device=device)

loss_history = []
t0 = time.time()

prior.train()
for epoch in range(1, PRIOR_EPOCHS+1):
    perm = torch.randperm(X_t.size(0), device=device)
    total_loss = 0.0
    for i in range(0, X_t.size(0), PRIOR_BATCH):
        idx = perm[i:i+PRIOR_BATCH]
        logits = prior(X_t[idx])
        loss = loss_fn(logits, y_t[idx])
        opt_prior.zero_grad()
        loss.backward()
        opt_prior.step()
        total_loss += loss.item() * idx.numel()
    avg_loss = total_loss / X_t.size(0)
    loss_history.append(float(avg_loss))
    print(f"Prior epoch {epoch}/{PRIOR_EPOCHS} loss={avg_loss:.4f}")

prior.eval()
print("Prior training seconds:", round(time.time()-t0, 2))

plt.figure(figsize=(6,3.2))
plt.plot(loss_history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("CE Loss")
plt.title("Variant 6: Prior training loss")
plt.grid(True)
plt.tight_layout()
plt.show()

## 9. Gen-Aug agent + evaluation

In [ ]:
class GenAugAgent(AgentRandon):
    def __init__(self, name, prior_model, eps=0.25, temperature=0.85):
        super().__init__(name=name)
        self.prior = prior_model
        self.eps = float(eps)
        self.temperature = float(temperature)

    def get_action(self, observation):
        # fallback exploration
        if np.random.rand() < self.eps:
            return super().get_action(observation)

        obs = np.asarray(observation, dtype=np.float32).ravel()
        obs_t = torch.tensor(obs, device=device).unsqueeze(0)

        with torch.no_grad():
            logits = self.prior(obs_t).squeeze(0).detach().cpu().numpy()

        mask = last200_mask(observation, n_actions=N_ACTIONS)
        if mask is None:
            return super().get_action(observation)

        valid = np.where(mask > 0.5)[0]
        if len(valid) == 0:
            return super().get_action(observation)

        v_logits = logits[valid] / max(self.temperature, 1e-6)
        probs = softmax_np(v_logits)
        a = int(np.random.choice(valid, p=probs))

        onehot = [0]*N_ACTIONS
        onehot[a] = 1
        return onehot

EPSILON_FALLBACK = 0.25
TEMP = 0.85

gen_scores, gen_mean, gen_std = eval_agent(
    lambda name: GenAugAgent(name, prior, eps=EPSILON_FALLBACK, temperature=TEMP),
    n_games=N_EVAL_GAMES
)
print("Gen-Aug mean:", gen_mean)
print("Gen-Aug std :", gen_std)
print("Δ mean:", gen_mean - baseline_mean)

plt.figure(figsize=(6,3.2))
plt.bar(["Baseline","Gen-Aug"], [baseline_mean, gen_mean], yerr=[baseline_std, gen_std], capsize=5)
plt.ylabel(f"Mean performance ({N_EVAL_GAMES} games)")
plt.title("Baseline vs Gen-Aug")
plt.grid(axis="y")
plt.tight_layout()
plt.show()

## 10. Save results.json + plots to disk

In [ ]:
os.makedirs("task2_outputs", exist_ok=True)

results = {
    "variant2_3_reinforce": {
        "train_games": int(N_TRAIN_GAMES),
        "lr": float(LR),
        "temperature": float(TEMPERATURE),
        "seed": int(SEED),
        "train_return_mean_last10": float(np.mean(train_returns[-10:])) if len(train_returns) >= 10 else float(np.mean(train_returns)),
    },
    "variant6_genaug": {
        "data_games": int(N_DATA_GAMES),
        "prior_epochs": int(PRIOR_EPOCHS),
        "prior_lr": float(PRIOR_LR),
        "epsilon_fallback": float(EPSILON_FALLBACK),
        "temperature": float(TEMP),
        "baseline_mean": float(baseline_mean),
        "baseline_std": float(baseline_std),
        "genaug_mean": float(gen_mean),
        "genaug_std": float(gen_std),
        "delta_mean": float(gen_mean - baseline_mean),
        "prior_loss_history": [float(x) for x in loss_history],
    }
}

with open("task2_outputs/results.json", "w") as f:
    json.dump(results, f, indent=2)

# save comparison plot
plt.figure(figsize=(6,3.2))
plt.bar(["Baseline","Gen-Aug"], [baseline_mean, gen_mean], yerr=[baseline_std, gen_std], capsize=5)
plt.ylabel(f"Mean performance ({N_EVAL_GAMES} games)")
plt.title("Baseline vs Gen-Aug")
plt.grid(axis="y")
plt.tight_layout()
plt.savefig("task2_outputs/comparison.png", dpi=160)
plt.close()

# save prior loss plot
plt.figure(figsize=(6,3.2))
plt.plot(loss_history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("CE Loss")
plt.title("Prior training loss")
plt.grid(True)
plt.tight_layout()
plt.savefig("task2_outputs/prior_loss.png", dpi=160)
plt.close()

print("✅ Saved: task2_outputs/results.json, comparison.png, prior_loss.png")
print("Files:", os.listdir("task2_outputs"))

## 11. Export PNGs + Download (Colab)

This section saves key plots as **high-quality PNGs** into `task2_png_outputs/`, then (optionally) downloads a ZIP in Colab.


In [ ]:
import os, shutil
import matplotlib.pyplot as plt
import numpy as np

os.makedirs("task2_png_outputs", exist_ok=True)

def save_png(fig_maker, path):
    try:
        fig_maker()
        plt.tight_layout()
        plt.savefig(path, dpi=300)
        plt.close()
        print("✅ Saved:", path)
    except Exception as e:
        plt.close()
        print("⚠ Skipped", path, "->", repr(e))

# 1) Training rewards
def _fig1():
    if 'train_returns' not in globals() or len(train_returns) == 0:
        raise ValueError("train_returns missing/empty")
    plt.figure(figsize=(7,4))
    plt.plot(train_returns)
    plt.xlabel("Training Game")
    plt.ylabel("Delayed Reward (Performance)")
    plt.title("Variant 2/3 — REINFORCE Training Rewards")
    plt.grid(True)

save_png(_fig1, "task2_png_outputs/training_rewards.png")

# 2) Evaluation curve
def _fig2():
    if 'eval_means' not in globals() or len(eval_means) == 0:
        raise ValueError("eval_means missing/empty")
    xs = [a for (a,_,_) in eval_means]
    ms = [b for (_,b,_) in eval_means]
    ss = [c for (_,_,c) in eval_means]
    plt.figure(figsize=(7,4))
    plt.errorbar(xs, ms, yerr=ss, marker="o", capsize=5)
    plt.xlabel("Training Game")
    plt.ylabel("Mean Evaluation Performance")
    plt.title("Evaluation Performance During Training")
    plt.grid(True)

save_png(_fig2, "task2_png_outputs/evaluation_curve.png")

# 3) Baseline vs Gen-Aug
def _fig3():
    required = ['baseline_mean','baseline_std','gen_mean','gen_std']
    if not all(k in globals() for k in required):
        raise ValueError("Baseline/Gen results missing")
    plt.figure(figsize=(6,4))
    plt.bar(["Baseline", "Gen-Aug"], [baseline_mean, gen_mean], yerr=[baseline_std, gen_std], capsize=6)
    plt.ylabel("Mean Performance")
    plt.title("Baseline vs Generative-Augmented Agent")
    plt.grid(axis="y")

save_png(_fig3, "task2_png_outputs/baseline_vs_genaug.png")

# 4) Prior loss
def _fig4():
    if 'loss_history' not in globals() or len(loss_history) == 0:
        raise ValueError("loss_history missing/empty")
    plt.figure(figsize=(6,4))
    plt.plot(loss_history, marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Cross Entropy Loss")
    plt.title("Variant 6 — Prior Training Loss")
    plt.grid(True)

save_png(_fig4, "task2_png_outputs/prior_loss.png")

print("\n📁 task2_png_outputs contains:", os.listdir("task2_png_outputs"))

# Optional: ZIP everything for easy upload/download
zip_name = shutil.make_archive("task2_png_outputs", "zip", "task2_png_outputs")
print("✅ Created ZIP:", zip_name)

# If running on Colab, download the ZIP (safe import)
try:
    from google.colab import files
    files.download(zip_name)
except Exception as e:
    print("ℹ Not running on Colab (or download not available):", repr(e))
